# GSB 5544 — Topic 12.2: Extraction Walkthrough — Building the Week 12 Data Files from Their Sources — SOLUTIONS
*Topic 12.1 starts from two cached CSV files. This read-along notebook shows where they came from and rebuilds them
live, one step at a time, with every question answered.*

> **Instructor note.** **Run of show (about 25 minutes, read together, projector on).** Part 0 the two sources (3 min) · Part 1 Wikipedia:
> fetch, find the table, look, save, compare (10 min) · Part 2 nflverse: one season, then the loop, check, save,
> compare (10 min) · Part 3 what the Extract step leaves behind (2 min).
> **Four kinds of question**, labelled in bold so you can pick as you go: *Explain, step by step* (a student reads a
> cell aloud line by line), *Write the code* (one or two lines; give them 60 seconds, then show the solution),
> *Fill in the blank* (`____` in code or prose) and *Discuss* (judgement; the written answer is in this notebook).
> **Needs the internet.** If the room's Wi-Fi fails, read from this executed solution; the outputs are the point.
> **The point to land:** an extraction is five things — a source you are allowed to use, the code that reads it,
> the rows and columns you keep, the saved file, and the date — and Topic 12.1 could start from one line of
> `pd.read_csv` only because all five were done here first.

## Part 0 — Where the Week 12 data came from

Topic 12.1 reads two files. Neither was typed in; each is the saved result of code that read a public source.

| File 12.1 reads | Source | What the source looks like | Permission |
|---|---|---|---|
| `hof_inductees.csv` (783 rows) | Wikipedia, [*List of Pro Football Hall of Fame inductees*](https://en.wikipedia.org/wiki/List_of_Pro_Football_Hall_of_Fame_inductees) | one web page, about 936 KB of HTML, 15 tables; the NFL table has one row per inductee **and team** | text licensed CC BY-SA 4.0; identify your script in the request |
| `nfl_rb_seasons_1999_2025.csv` (4,619 rows) | nflverse-data on GitHub, release [`stats_player`](https://github.com/nflverse/nflverse-data/releases/tag/stats_player) | one CSV per season, `stats_player_reg_<season>.csv`, about 2,000 players × 148 columns each | data released CC BY 4.0 by the nflverse project |

The outline's first choice, Pro Football Reference, was dropped: its terms restrict automated access and the use of
its content for machine-learning methods. **Checking permission is part of the Extract step**, before any code.

An extraction is finished when five things exist: the source (a URL you may use), the **code** that reads it, the
rows and columns kept, the saved **file**, and the access **date**. This notebook produces all five for both files.
It writes into a folder of its own, `week12_data/`, and at the end compares what it built with the repository's
cached copies.

**Discuss.** Topic 12.1 reads each file with one line of `pd.read_csv`. What is lost if a project *only* has that
line and the file, and not this notebook?

**Answer:** Nobody can say where the numbers came from, whether the source allowed the use, which rows were dropped on the way
in, or when it was read — so nobody can repeat it or update it. The file alone is a claim; the code plus the date
is evidence. (It is also the first thing a reader of your final project will ask: *where did this come from?*)

In [1]:
import io
import warnings
from datetime import date
from pathlib import Path

import pandas as pd
import requests

warnings.filterwarnings("ignore")

OUT = Path("week12_data")            # this notebook writes here; the repository's copies are not touched
OUT.mkdir(exist_ok=True)
CACHE = "https://raw.githubusercontent.com/gato365/gsb5544_instructor_learn_prep/main/assignments/Data/"   # the cached copies Topic 12.1 reads
print("writing to", OUT.resolve())

writing to /private.../week12_data


## Part 1 — Source 1: the Hall of Fame list (Wikipedia)

### 1.1 Fetch the page

A web page is text. `requests.get` asks the server for it and hands back a *response*; the HTML is in `.text`.

In [2]:
HOF_URL = "https://en.wikipedia.org/wiki/List_of_Pro_Football_Hall_of_Fame_inductees"

response = requests.get(HOF_URL, headers={"User-Agent": "GSB5544-classroom-notebook/1.0"}, timeout=60)
print(response.status_code, "|", f"{len(response.text):,}", "characters of HTML")
response.text[:300]

200 | 936,065 characters of HTML


'<!DOCTYPE html>\n<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disab'

**Explain, step by step.** Read the `requests.get(...)` line aloud: what are the three arguments, what does the
status code tell you, and what is in `response.text`?

**Answer:** `HOF_URL` is the address to fetch. `headers={"User-Agent": ...}` names the program making the request — Wikipedia
asks automated clients to identify themselves, and a descriptive name is the polite minimum. `timeout=60` makes
the call fail after 60 seconds instead of hanging forever. A status code of `200` means "OK, here is the page"
(a `404` would mean the page does not exist, `429` that we are asking too often). `response.text` is the whole
page as one string of HTML — the first 300 characters are the `<!DOCTYPE html>` boilerplate, nothing like a table
yet.

### 1.2 From HTML to tables

`pd.read_html` finds every `<table>` in a page and turns each into a DataFrame. It accepts a file or a file-like
object, so the string is wrapped in `io.StringIO`, which makes a string behave like an open **file**.

In [3]:
tables = pd.read_html(io.StringIO(response.text))

In [4]:
# Write the code: print how many tables the page has, then the shape of each one
print(len(tables), "tables")
[t.shape for t in tables]

15 tables


[(0, 7),
 (783, 5),
 (156, 6),
 (9, 4),
 (3, 4),
 (1, 4),
 (3, 4),
 (1, 4),
 (12, 2),
 (19, 4),
 (4, 4),
 (1, 4),
 (4, 4),
 (1, 4),
 (6, 2)]

15 tables, most of them navigation boxes with a handful of rows. Two are large.

### 1.3 Find the right table by its column names, not its position

The inductee table could be `tables[1]` today and `tables[2]` after the next edit to the page. A test on the
**column names** fails loudly if the page changes; an index fails silently.

In [5]:
want = {"Inductee", "Class", "Position", "Team(s)", "Years"}
hits = [i for i, t in enumerate(tables) if want.issubset(t.columns)]
print("tables with those five columns:", hits)
for i in hits:
    print(i, tables[i].shape, list(tables[i].columns))

tables with those five columns: [1, 2]
1 (783, 5) ['Inductee', 'Class', 'Position', 'Team(s)', 'Years']
2 (156, 6) ['Inductee', 'Class', 'Position', 'League(s)', 'Team(s)', 'Years']


**Explain, step by step.** Read the `hits = [...]` line aloud. What does `enumerate` give, what does `issubset`
test, and why is the result a list of numbers?

**Answer:** `enumerate(tables)` yields pairs `(i, t)`: the position of each table and the table itself. `want.issubset(t.columns)`
asks whether every name in the set `want` appears among that table's columns — a set test, so the order of the
columns does not matter and extra columns are allowed. The comprehension keeps the `i` of every table that passes,
so `hits` is the list of table numbers that *could* be the one we want: here 1 and 2.

Two tables pass: table 1 (NFL inductees) and table 2 (156 rows of pre-NFL and
other-league careers, with a `League(s)` column). The project wants the NFL table, so the extra column is the
tie-breaker.

In [6]:
nfl_table = [tables[i] for i in hits if "League(s)" not in tables[i].columns][0]
hof = nfl_table[["Inductee", "Class", "Position", "Team(s)", "Years"]]
hof.shape

(783, 5)

**Discuss.** Why select the five columns by name again, when the table already has exactly those five?

**Answer:** Because it will not always. Selecting by name fixes the column order, drops anything the page adds later (a new
"Ref." column, say) and raises a `KeyError` if one of the five disappears. Topic 12.1 depends on exactly these
five names; this line is where that promise is made.

### 1.4 Look before you save

Saving is cheap; saving the wrong thing is expensive. Look at the head, the types and the repeats.

In [7]:
print(hof.dtypes, "\n")
hof.head(8)

Inductee      str
Class       int64
Position      str
Team(s)       str
Years         str
dtype: object 



,Inductee,Class,Position,Team(s),Years
0,Herb Adderley,1980,Cornerback,Green Bay Packers,1961–1969
1,Herb Adderley,1980,Cornerback,Dallas Cowboys,1970–1972
2,Troy Aikman**,2006,Quarterback,Dallas Cowboys,1989–2000
3,Eric Allen,2025,Cornerback,Philadelphia Eagles,1988–1994
4,Eric Allen,2025,Cornerback,New Orleans Saints,1995–1997
5,Eric Allen,2025,Cornerback,Oakland Raiders,1998–2001
6,George Allen[3],2002,Coach,Los Angeles Rams,1966–1970
7,George Allen[3],2002,Coach,Washington Redskins,1971–1977


In [8]:
# Write the code: count the rows, the distinct inductees, and show the five most repeated names
print(len(hof), "rows;", hof["Inductee"].nunique(), "distinct inductees")
hof["Inductee"].value_counts().head(5)

783 rows; 387 distinct inductees


Inductee
Walt Kiesling      9
Jim Thorpe §       9
Jimmy Conzelman    8
Joe Guyon          7
Guy Chamberlin     6
Name: count, dtype: int64

783 rows but 387 people: one row per inductee **and team**, up to 9 rows for one
person. Names carry footnote marks (`Marcus Allen** [4]`), and `Position` mixes "Running back", "Halfback" and
long pre-modern labels. **None of that is fixed here.** The Extract step saves what the source said; cleaning
names, parsing years and collapsing to one row per person is **Transform** work and lives in Topic 12.1, Part 3,
where every change is visible and checkable.

### 1.5 Save it, with the date

In [9]:
hof.to_csv(OUT / "hof_inductees.csv", index=False)
(OUT / "nfl_hof_access_date.txt").write_text(date.today().isoformat() + "\n")
print("saved", OUT / "hof_inductees.csv", "| accessed", (OUT / "nfl_hof_access_date.txt").read_text().strip())

saved week12_data/hof_inductees.csv | accessed 2026-11-10


**Fill in the blank (prose).** Without `index=False`, re-reading the CSV would show an extra column called
**Unnamed: 0** holding the old row numbers.

In [10]:
# Write the code: re-read the saved file and check that it has the same shape as `hof`
pd.read_csv(OUT / "hof_inductees.csv").shape == hof.shape

True

### 1.6 Compare with the cached copy that Topic 12.1 reads

The repository's copy was saved on 2026-11-10. Wikipedia changes; the comparison says whether today's page
still agrees with it.

In [11]:
cached_hof = pd.read_csv(CACHE + "hof_inductees.csv")
same = hof.reset_index(drop=True).astype(str).equals(cached_hof.astype(str))
print("live", hof.shape, "| cached", cached_hof.shape, "| identical:", same)

live (783, 5) | cached (783, 5) | identical: True


**Discuss.** Suppose this prints `False` next August. What happened, which copy should the project use, and what
does "refresh the data" have to include?

**Answer:** A new class was inducted, or someone edited a name, a team or a footnote. The project keeps using the **cached**
copy with its recorded date, so every number in the notebook stays reproducible. Refreshing is a deliberate act:
re-run the extraction, write the new file *and* the new date, then re-run every downstream cell and re-read every
sentence that quotes a count, because some of them will have changed. (The repository does this with
`tools/build_week12_project_exploration.py --refresh-data`.)

## Part 2 — Source 2: season-by-season statistics (nflverse)

### 2.1 One file per season

The nflverse project publishes one CSV per season on a GitHub *release* page. The file names differ only in the
season, so one URL pattern with a `{season}` slot covers all of them.

In [12]:
NFLVERSE_URL = "https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_reg_{season}.csv"
print(NFLVERSE_URL.format(season=2024))

one = pd.read_csv(NFLVERSE_URL.format(season=2024), low_memory=False)
print(one.shape)
list(one.columns)[:30]

https://github.com/nflverse/nflverse-data/releases/download/stats_player/stats_player_reg_2024.csv


(1997, 148)

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'season_type',
 'recent_team',
 'games',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40']

**Explain, step by step.** What does `.format(season=2024)` do to the string? Why read the file with
`pd.read_csv` straight from a URL? What is `low_memory=False` for?

**Answer:** `.format` replaces the `{season}` slot with `2024`, giving the exact file name on the release page. `pd.read_csv`
accepts a URL as well as a path, so no separate download step is needed — pandas fetches the file and parses it in
one call. `low_memory=False` makes pandas read the whole file before deciding each column's type; with the default,
a large file is read in chunks and a column that looks numeric in one chunk and text in another produces a
"mixed types" warning and an `object` column.

1,997 players and 148 columns for 2024 alone — passing, rushing, receiving, defence, kicking, punting,
fantasy points. The project needs a dozen of them.

In [13]:
# Write the code: show which positions appear in the 2024 file and how many players each has
one["position"].value_counts()

position
LB     266
WR     236
CB     201
DT     152
RB     148
DE     146
OT     129
TE     127
SAF    116
G       93
QB      78
DB      57
K       43
C       41
P       39
LS      27
FS      20
OLB     19
MLB     12
S       11
ILB     10
FB       8
NT       8
DL       5
OL       4
Name: count, dtype: int64

### 2.2 Which rows, which columns

Two decisions, made once and written down: keep the players whose listed position is a running back, and keep
the identifying columns plus the rushing and receiving counts.

In [14]:
KEEP = ["player_id", "player_display_name", "position", "recent_team", "season", "games",
        "carries", "rushing_yards", "rushing_tds", "receptions", "targets", "receiving_yards", "receiving_tds"]

is_back = one["position"].isin(["RB", "HB", "FB"])
one_rb = one.loc[is_back, KEEP]
print(one_rb.shape)
one_rb.sort_values("rushing_yards", ascending=False).head(5)

(156, 13)


,player_id,player_display_name,position,recent_team,season,games,carries,rushing_yards,rushing_tds,receptions,targets,receiving_yards,receiving_tds
540,00-0034844,Saquon Barkley,RB,PHI,2024,16,345,2005,13,33,43,278,2
199,00-0032764,Derrick Henry,RB,BAL,2024,17,325,1921,16,19,22,193,2
1465,00-0038542,Bijan Robinson,RB,ATL,2024,17,304,1456,14,61,72,431,1
803,00-0036223,Jonathan Taylor,RB,IND,2024,14,303,1431,11,18,31,136,1
1680,00-0039139,Jahmyr Gibbs,RB,DET,2024,17,250,1412,16,52,63,517,4


**Explain, step by step.** Read `one_rb = one.loc[is_back, KEEP]` aloud. What is `is_back`, and what do the two
things inside the square brackets each select?

**Answer:** `is_back` is a Series of `True` / `False`, one per player, `True` when the position is one of the three labels.
`.loc[rows, columns]` takes a row selector and a column selector: the boolean Series keeps only the `True` rows
(156 of 1,997 for 2024: 148 RB and 8 FB), and the list `KEEP` keeps only those 13 columns,
in that order. The result is a new, smaller DataFrame; `one` is unchanged.

**Discuss.** Three choices to defend: (a) `player_id` is kept although `player_display_name` is there; (b)
`receptions`, `targets` and `receiving_yards` are kept although Topic 12.1 only plots rushing; (c) `"HB"` is in
the list although no 2024 player has it.

**Answer:** (a) Names are not identifiers: two different players are called Adrian Peterson and two Ricky Williams. The id is
the key that Topic 12.1's crosswalk joins on; drop it here and the join can never be made right later.
(b) The project question promises "rushing *and receiving*", and keeping three columns now costs nothing, while
going back to the source for them later means redoing the extraction. Keep what the questions might need, no more.
(c) Position vocabularies change between seasons, and the list is a *rule*, not a description of one file;
writing `HB` costs nothing and the check in 2.4 shows which labels actually occurred (FB and RB, never HB).
The same rows come from `position_group == "RB"`; either is fine, as long as the notebook says which was used.

### 2.3 All 27 seasons: the loop

The same three lines, once per season, collected in a list and stacked at the end. One line in the loop is a
**check**: if a season's file does not have the 13 columns, stop immediately and say which season and which
columns.

In [15]:
FIRST_SEASON, LAST_SEASON = 1999, 2025

frames = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    d = pd.read_csv(NFLVERSE_URL.format(season=season), low_memory=False)
    missing = [c for c in KEEP if c not in d.columns]
    assert not missing, (season, missing)          # stop loudly if a season's file has changed shape
    frames.append(d.loc[d["position"].isin(["RB", "HB", "FB"]), KEEP])
    print(season, f"{len(d):,} players ->", len(frames[-1]), "backs")

rb = pd.concat(frames, ignore_index=True)
rb.shape

1999 1,678 players -> 174 backs


2000 1,647 players -> 169 backs


2001 1,660 players -> 169 backs


2002 1,707 players -> 179 backs


2003 1,736 players -> 185 backs
2004 1,780 players -> 186 backs


2005 1,779 players -> 181 backs


... (14 outputs left out)

2022 2,007 players -> 169 backs
2023 1,943 players -> 158 backs


2024 1,997 players -> 156 backs


2025 2,020 players -> 157 backs


(4619, 13)

**Explain, step by step.** Read the loop aloud, one line at a time: the `range`, the read, the `missing` list,
the `assert`, the `append`, the `print`, and the `concat` after the loop.

**Answer:** `range(1999, 2026)` produces 1999 to 2025 — the end is excluded, hence `LAST_SEASON + 1`. Each pass reads that
season's file into `d`. `missing` lists the names in `KEEP` that are *not* columns of `d`; `assert not missing`
passes when the list is empty and otherwise stops the notebook with the season and the missing names in the error,
so a changed file can never be stacked silently. `frames.append(...)` keeps that season's running-back rows and
13 columns; `print` shows the row counts as they go, which is how you notice a season with zero backs. After the
loop, `pd.concat(frames, ignore_index=True)` stacks the 27 small tables into one and renumbers the rows 0 to
4,619 instead of restarting at 0 for each season.

### 2.4 Check before saving

One row per player per season is the assumption every later step rests on. Say it as an `assert`, and look at the
counts.

In [16]:
assert not rb.duplicated(["player_id", "season"]).any()      # one row per player per season
print(rb["player_id"].nunique(), "distinct players |", rb["season"].nunique(), "seasons")
rb["position"].value_counts()

1169 distinct players | 27 seasons


position
RB    3903
FB     716
Name: count, dtype: int64

In [17]:
# Write the code: count the backs in each season and print the smallest and largest counts
per_season = rb.groupby("season").size()
print("backs per season:", per_season.min(), "to", per_season.max())
per_season

backs per season: 156 to 186


season
1999    174
2000    169
2001    169
2002    179
2003    185
2004    186
2005    181
2006    180
2007    172
2008    178
2009    173
2010    168
2011    164
2012    174
2013    172
2014    166
2015    173
2016    173
2017    165
2018    164
2019    161
2020    173
2021    180
2022    169
2023    158
2024    156
2025    157
dtype: int64

**Discuss.** Fullbacks fall from 34 in 1999 to 6 in 2025. Does the extraction have a problem, and
what does the trend mean for Topic 12.1's 50-carry rule?

**Answer:** The extraction is fine: it kept every player the source *labels* RB or FB, and the label is a roster designation,
not a usage count. The trend is the game changing — fewer teams carry a fullback — and it is why Topic 12.1 defines
the comparison group by **usage** (at least 50 carries) rather than by label: a 2025 fullback with 6 carries and a
2025 running back with 6 carries are the same kind of row, and both drop out. The label stays in the file so the
rule can be revisited (Figure 4 in 12.1 does exactly that).

### 2.5 Save, and compare with the cached copy

In [18]:
rb.to_csv(OUT / "nfl_rb_seasons_1999_2025.csv", index=False)

cached_rb = pd.read_csv(CACHE + "nfl_rb_seasons_1999_2025.csv")
compare = pd.DataFrame({"live_rows":    rb.groupby("season").size(),
                        "cached_rows":  cached_rb.groupby("season").size(),
                        "live_yards":   rb.groupby("season")["rushing_yards"].sum(),
                        "cached_yards": cached_rb.groupby("season")["rushing_yards"].sum()})
differs = compare[(compare["live_rows"] != compare["cached_rows"]) | (compare["live_yards"] != compare["cached_yards"])]
print("live", rb.shape, "| cached", cached_rb.shape, "| seasons that differ:", len(differs))
differs

live (4619, 13) | cached (4619, 13) | seasons that differ: 0


,live_rows,cached_rows,live_yards,cached_yards
season,,,,


**Discuss.** The 2026 season is being played as you read this, yet the loop stops at 2025. Why not include the
current season, and what has to change when it ends?

**Answer:** A season in progress changes every week, so a file read today and one read next Tuesday disagree, and a
percentile among *partial* seasons compares players who have played different numbers of games. Keep completed
seasons only. When 2026 ends: set `LAST_SEASON = 2026`, re-run this extraction, save a new file **with a new name**
(`..._1999_2026.csv`) and a new date, then re-run Topic 12.1 and re-read every sentence that quotes a number.

## Part 3 — What the Extract step leaves behind

In [19]:
sorted(p.name for p in OUT.iterdir())

['hof_inductees.csv',
 'nfl_hof_access_date.txt',
 'nfl_rb_seasons_1999_2025.csv']

Three files: the two data sets and the access date. Together with this notebook they *are* the Extract step; Topic
12.1 can begin with `pd.read_csv` because they exist.

| What | Where | Why it matters |
|---|---|---|
| the source and its permission | Part 0 | someone else can judge whether the use was allowed |
| the code that read it | Parts 1 and 2 | the extraction can be repeated on purpose, not by accident |
| the rows and columns kept | 1.3 and 2.2 | the reader knows what was dropped on the way in |
| the saved files | `week12_data/` | every later cell reads the same bytes |
| the access **date** | `nfl_hof_access_date.txt` | "the page said so on 10 November" is a checkable claim |

**Discuss (closing).** Where does E end and T begin? The filter to running backs and the 13 columns happened
*here*; stripping `**` from the names and splitting `1993–1995` into two years happens in Topic 12.1. Why draw the
line there?

**Answer:** Rule of thumb: **Extract saves what the source said; Transform changes it.** Choosing rows and columns does not
alter a single value, so it belongs with the extraction (and is documented there). Rewriting a name, parsing a
year or collapsing several rows into one *does* alter values, and every such change should be visible in the
analysis notebook where it can be checked against the raw file. If the cleaning were done here, Topic 12.1's
reader could never see what `Marcus Allen** [4]` looked like before it became `Marcus Allen`.

---
## For your own final project — the Extract checklist

1. **Permission first.** Read the source's terms before writing code; write one sentence in the notebook saying
   what the terms allow. If the answer is no, find another source (the outline's first choice was replaced for
   exactly this reason).
2. **Identify yourself and fail fast.** A descriptive `User-Agent` and a `timeout` on every request.
3. **Find things by name, not by position.** Tables by their columns, files by a pattern, columns by a list.
4. **Look before saving.** `shape`, `dtypes`, `head`, repeats of the key.
5. **Check with `assert`.** One row per key; the columns you expect; stop loudly otherwise.
6. **Save what the source said** — no cleaning in the Extract step — with `index=False` and an access date.
7. **Compare with the cache** when you refresh, and re-read every sentence that quotes a number.